<a href="https://colab.research.google.com/github/mostofa89/Pytorch_Fundamentals/blob/main/Pytorch_Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
torch.__version__

# Linear Regression
* Y = wx + b
* w is weight
* b is bias
* x is the data points

In [ ]:
w = 0.7
b = 0.3

start = 0
end = 1
step = 0.02

X = torch.arange(start, end, step).unsqueeze(dim = 1)

y = w * X + b

print(len(X), len(y))
for i in range(len(X)):
  print(X[i], y[i])

##Spliting into Train and Test

In [ ]:
train_split = int(.8 * len(X))

X_train, y_train = X[ : train_split], y[ : train_split]
X_test, y_test = X[train_split : ], y[train_split : ]

print(len(X_train), len(y_train), len(X_test), len(y_test))

##Ploting The Predictions

In [ ]:
def plot_prediction(train_data=X_train, train_label=y_train,
                    test_data=X_test, test_label=y_test,
                    predictions=None):

  plt.scatter(train_data, train_label, c='b', s=10, label='Training data')
  plt.scatter(test_data, test_label, c='g', s=10, label='Testing data')

  if predictions is not None:
    plt.scatter(test_data, predictions, c='r', s=10, label='Predictions')

  plt.legend()
  plt.show()

plot_prediction()

## Linear Regression Model

In [ ]:
class LinearRegression(nn.Module): # pytorch import everything from pytorch NN module
  def __init__(self):
    super().__init__()
    self.weight = nn.Parameter(torch.randn(1, requires_grad=True, dtype=torch.float)) # requires_grad=True means that i will need gradient descent
    self.bias = nn.Parameter(torch.randn(1, requires_grad=True, dtype=torch.float))


  def forward(self, X:torch.Tensor):
    return self.weight * X + self.bias


In [ ]:
torch.manual_seed(42) # Smae as the random state 42
torch.randn(1)

In [ ]:
torch.manual_seed(42)

model_0 = LinearRegression()

print(list(model_0.parameters()))
# module_0

In [ ]:
model_0.state_dict()

In [ ]:
X_test, y_test

In [ ]:
y_pred = model_0(X_test)

y_pred

In [ ]:
with torch.inference_mode(): # It is advance
  y_pred = model_0(X_test)

# with torch.no_grad():
#   y_pred = model_0(X_test)

y_pred

In [ ]:
plot_prediction(predictions=y_pred)

In [ ]:
loss_func = nn.L1Loss()

optimizer = torch.optim.SGD(params=model_0.parameters(), lr = 0.01)

In [ ]:
list(model_0.parameters())

## Training and Testing Loop
* Loop Through The Data
* Forward Pass / Forward Propagation
* Calculate the Loss
* Optimizer Zero Grad
* Loss Backward --> Calculate the Gradient / **Backpropagation**
* Optimizer Step / **Gradient Descent**

In [ ]:
torch.manual_seed(42)
epochs = 200

#Track Different values
epoch_count = []
loss_values = []
test_loss_values = []

for epoch in range(epochs):
  model_0.train()

  # Forward Pass
  y_pred = model_0(X_train)

  # Calulate The Loss
  loss = loss_func(y_pred, y_train)
  # print(f"Loss is {loss}")
  # Optimizer Zero Grad
  optimizer.zero_grad()

  # Backpropagation
  loss.backward()

  # Step of The Opimizer
  optimizer.step()

  model_0.eval()

  with torch.inference_mode():
    test_pred = model_0(X_test)

    test_loss = loss_func(test_pred, y_test)

  if epoch % 10 == 0:
    epoch_count.append(epoch)
    loss_values.append(loss)
    test_loss_values.append(test_loss)

    print(f"Epochs : {epoch} | Loss : {loss} | Test Loss : {test_loss}")

  # printing state dict
  # print(model_0.state_dict())

In [ ]:
with torch.inference_mode():
  y_pred_new = model_0(X_test)

In [ ]:
model_0.state_dict()

In [ ]:
print(np.array(torch.tensor(loss_values).cpu().numpy()))

In [ ]:
plt.plot(epoch_count, np.array(torch.tensor(loss_values).cpu().numpy()), label = 'Train Loss')
plt.plot(epoch_count, np.array(torch.tensor(test_loss_values).cpu().numpy()), label = 'Test Loss')

plt.title('Train and Test Loss Curves')
plt.xlabel('epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
plot_prediction(predictions=y_pred_new)

##Saving and Loading The Model
* Using torch.save()
* Using torch.load()

In [ ]:
from pathlib import Path

# Create Model directory
MODEL_PATH = Path('model')
MODEL_PATH.mkdir(parents = True, exist_ok = True)

# Model save path
MODEL_NAME = "Single_Variable_Linear_Regression"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

#Saving The Model
print(f"Saving The Model to: {MODEL_SAVE_PATH}")
torch.save(model_0.state_dict(), f=MODEL_SAVE_PATH)

In [ ]:
!ls -l model